In [ ]:
# SETUP KAGGLE 
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# INSTALASI LIBRARY (DENGAN PERBAIKAN SCIPY) 
print("[INFO] Memperbaiki Konflik Numpy-Scipy...")
!pip install numpy scipy scikit-learn --upgrade --force-reinstall --quiet

print("[INFO] Menginstall Sentence-Transformers...")
!pip install sentence-transformers safetensors rank_bm25 Sastrawi --upgrade --quiet

# DETEKSI PATH INPUT
INPUT_ROOT = "/kaggle/input"
DATASET_PATH = None

print(f"\n[INFO] Mencari Dataset di {INPUT_ROOT}...")
if os.path.exists(INPUT_ROOT):
    for dirname in os.listdir(INPUT_ROOT):
        if dirname != ".":
            print(f"   Ditemukan Folder: {dirname}")
            DATASET_PATH = os.path.join(INPUT_ROOT, dirname)
            break

if not DATASET_PATH:
    print("[FATAL] Dataset tidak ditemukan! Pastikan Anda sudah upload dataset zip & csv.")
    sys.exit()

print(f"[INFO] Path Dataset Terdeteksi: {DATASET_PATH}")

# CEK GPU
import torch
print(f"\n[INFO] Status GPU: {'AKTIF' if torch.cuda.is_available() else 'MATI'}")

[INFO] Memperbaiki Konflik Numpy-Scipy...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
mkl-umath 0.1.1 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.3.5 which is incompatible.
mkl-random 1.2.4 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.3.5 which is incompatible.
mkl-fft 1.3.8 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.3.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.5 which is incompatible.
datasets 4.4.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
ydata-profiling 4.17.0 requires numpy<2.2,>=1.16.0, but you have numpy 2.3.5 which is incompatible.
ydata-profiling 4.17.0 requires scipy<1.16,>=1.4.1, but you have scipy 1.16.3 which is incompatible.
category-encoders 2

In [ ]:
import os

start_path = '/kaggle/input'
found_path = None

for root, dirs, files in os.walk(start_path):
    # Kita cari tanda-tanda kehidupan model (config.json)
    if 'config.json' in files and 'pytorch_model.bin' in files:
        found_path = root
        break
    elif 'config.json' in files and 'model.safetensors' in files: 
        found_path = root
        break

if found_path:
    print(f"\nGanti MODEL_PATH kamu jadi:\nMODEL_PATH = '{found_path}'")
else:
    print("\n Cek upload-an kamu!")

In [ ]:

#  MINING 3 HARD NEGATIVES (SKIP TOP-5)

TOP_K_SEARCH = 20 
SKIP_N = 5
TAKE_N = 3  # Target kita: Ambil 3 Hard Negative

print(f"\n⛏️ Mining {TAKE_N} Hard Negatives per Query (Skip Top-{SKIP_N})...")
print(f"   Target: Rank 6, 7, dan 8.")

triplets = []
stats_skip = 0
stats_found_full = 0 # Menghitung query yang berhasil dapat full 3 kandidat

for idx, row in tqdm(df_clean.iterrows(), total=len(df_clean)):
    query_text = row['query']
    pos_text = row['positive']
    query_vec = model.encode(query_text, convert_to_tensor=True)
    hits = util.semantic_search(query_vec, corpus_embeddings, top_k=TOP_K_SEARCH)[0]
    true_id = text_to_id.get(pos_text)
    candidates = [h for h in hits if h['corpus_id'] != true_id]
    needed = SKIP_N + TAKE_N
    
    if len(candidates) >= needed:
        for i in range(TAKE_N):
            selected = candidates[SKIP_N + i] 
            
            neg_text = unique_tafsirs[selected['corpus_id']]
            score = selected['score']
            
            triplets.append({
                'query': query_text,
                'pos': pos_text,
                'neg': neg_text,
                'sbert_score': score,
                'rank': SKIP_N + i + 1 # Info rank (6, 7, atau 8)
            })
        stats_found_full += 1
    else:
        stats_skip += 1

# SAVE RESULT
df_final = pd.DataFrame(triplets)

print("HASIL AKHIR (VERSI 3 HARD NEGATIVES)")
print(f"Total Query Awal   : {len(df_clean)}")
print(f"Total Triplets     : {len(df_final)} (Harusnya ~3x Query Awal)")
print(f"Query Sukses (Full): {stats_found_full}")
print(f"Query Di-skip      : {stats_skip} (Kurang kandidat)")
print("-" * 30)

filename = 'dataset_triplet_sbert_mining_3neg.csv'
df_final.to_csv(filename, index=False)
print(f"File Tersimpan: {filename}")

# Preview Data (Cek apakah 1 query punya 3 baris)
print("\n Preview (Perhatikan Query yang sama muncul 3 kali):")
print(df_final[['query', 'sbert_score', 'rank']].head(6))


⛏️ Mining 3 Hard Negatives per Query (Skip Top-5)...
   Target: Rank 6, 7, dan 8.


  0%|          | 0/42593 [00:00<?, ?it/s]


HASIL AKHIR (VERSI 3 HARD NEGATIVES)
Total Query Awal   : 42593
Total Triplets     : 127779 (Harusnya ~3x Query Awal)
Query Sukses (Full): 42593
Query Di-skip      : 0 (Kurang kandidat)
------------------------------
💾 File Tersimpan: dataset_triplet_sbert_mining_3neg.csv

🔍 Preview (Perhatikan Query yang sama muncul 3 kali):
                                               query  sbert_score  rank
0  Mengapa memulai pekerjaan dengan bismillah itu...     0.329835     6
1  Mengapa memulai pekerjaan dengan bismillah itu...     0.327251     7
2  Mengapa memulai pekerjaan dengan bismillah itu...     0.319955     8
3  Apa makna dan keutamaan membaca basmalah sebel...     0.341774     6
4  Apa makna dan keutamaan membaca basmalah sebel...     0.332728     7
5  Apa makna dan keutamaan membaca basmalah sebel...     0.332452     8


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
from tqdm.auto import tqdm

# Load Data Triplet
INPUT_CSV = 'dataset_triplet_sbert_mining_3neg.csv' # Hasil mining 3-neg tadi
df_triplet = pd.read_csv(INPUT_CSV)

print(f"Total Triplet Awal: {len(df_triplet)}")

# Extract Positives (Hati-hati Duplikasi!)
# Ambil kolom Query dan Pos
df_pos = df_triplet[['query', 'pos']].copy()
df_pos.columns = ['query', 'text']
df_pos['label'] = 1

# PENTING: Hapus duplikat karena 1 query muncul 3 kali di triplet
df_pos = df_pos.drop_duplicates()
print(f"Total Data Positif (Unique): {len(df_pos)}")

# 3. Extract Negatives
# Ambil kolom Query dan Neg
df_neg = df_triplet[['query', 'neg']].copy()
df_neg.columns = ['query', 'text']
df_neg['label'] = 0
# Negatif tidak perlu drop duplicate karena isinya beda-beda (Rank 6,7,8)
print(f"Total Data Negatif: {len(df_neg)}")

# 4. Gabungkan (Concatenate)
df_train = pd.concat([df_pos, df_neg], ignore_index=True)

# Acak urutannya biar Label 1 dan 0 tidak berkelompok
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total Dataset Siap Training: {len(df_train)}")
print(df_train['label'].value_counts())

# FEATURE ENGINEERING (SBERT SIMILARITY)
print("Menghitung Fitur SBERT Cosine Similarity...")

# Encode secara batch biar cepat (JANGAN PAKAI LOOP APPLY SATU-SATU, LAMBAT!)
# Kita butuh embedding query dan embedding text
batch_size = 64

# Encode Queries
queries = df_train['query'].tolist()
q_embs = model.encode(queries, batch_size=batch_size, show_progress_bar=True, convert_to_tensor=True)

# Encode Texts (Candidate Tafsir)
texts = df_train['text'].tolist()
t_embs = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_tensor=True)

# Hitung Cosine Similarity
# util.cos_sim menghitung matriks n x n, kita cuma butuh diagonal (pasangan i dengan i)
# Cara efisien: (A . B) / (|A| * |B|) -> sudah dinormalisasi sbert, jadi tinggal dot product
cosine_scores = util.pairwise_cos_sim(q_embs, t_embs)

# Masukkan ke DataFrame
df_train['sbert_sim'] = cosine_scores.cpu().numpy()

# ==========================================
# 6. SIMPAN FINAL DATASET
# ==========================================
OUTPUT_FILE = 'dataset_xgboost_ready.csv'
df_train.to_csv(OUTPUT_FILE, index=False)

print(f"Dataset XGBoost tersimpan: {OUTPUT_FILE}")
print(df_train.head())

Total Triplet Awal: 127779
Total Data Positif (Unique): 42593
Total Data Negatif: 127779
Total Dataset Siap Training: 170372
label
0    127779
1     42593
Name: count, dtype: int64
Menghitung Fitur SBERT Cosine Similarity...


Batches:   0%|          | 0/2663 [00:00<?, ?it/s]

Batches:   0%|          | 0/2663 [00:00<?, ?it/s]

Dataset XGBoost tersimpan: dataset_xgboost_ready.csv
                                               query  \
0  Mengapa orang kafir mendapat azab yang keras d...   
1  Mengapa semua tumbuhan dan pepohonan tunduk ke...   
2  Dalil mengenai tantangan kaum musyrik kepada R...   
3  Mengapa manusia hanya diberi sedikit pengetahu...   
4  Bagaimana sikap Islam ketika kita tidak mampu ...   

                                                text  label  sbert_sim  
0  Kekalahan mereka di dunia bukan akhir segalany...      0   0.482377  
1  dan tetumbuhan tak berbatang dan pepohonan ber...      1   0.746225  
2  Seandainya orang kafir itu mengetahui dengan p...      0   0.445828  
3  Dan demikianlah Kami wahyukan kepadamu, wahai ...      0   0.350140  
4  Maka, pada saat datang azab kepada mereka seca...      0   0.361543  


In [ ]:
# TAHAP 1: FEATURE ENGINEERING 
!pip install -q rank_bm25 nltk

import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
import string
import nltk
from nltk.corpus import stopwords
from tqdm.auto import tqdm

# 1. SETUP STOPWORDS
try:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('indonesian'))
    print("Stopwords Indonesia loaded.")
except:
    print("Download NLTK gagal. Menggunakan list manual.")
    stop_words = set(['yang', 'dan', 'di', 'ke', 'dari', 'ini', 'itu', 'untuk', 'kepada', 'pada', 'adalah', 'sebagai', 'bagi', 'dengan', 'atau', 'bahwa'])

# 2. LOAD DATA
INPUT_FILE = 'dataset_xgboost_ready.csv'
df = pd.read_csv(INPUT_FILE)
print(f"Memproses {len(df)} baris data...")

# 3. PREPROCESSING (CLEANING)
def clean_tokens(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    return [w for w in tokens if w not in stop_words] # Buang stopwords

print("Tokenizing & Cleaning (Removing Stopwords)...")
df['q_tokens'] = df['query'].apply(clean_tokens)
df['t_tokens'] = df['text'].apply(clean_tokens)

# 4. HITUNG FITUR LEKSIKAL
print("Menghitung Overlap, Jaccard, & Length Ratio...")

# Length Features
df['len_q'] = df['q_tokens'].apply(len)
df['len_t'] = df['t_tokens'].apply(len)
df['len_ratio'] = df['len_q'] / (df['len_t'] + 1e-9)

# Overlap & Jaccard
def calc_metrics(row):
    set_q = set(row['q_tokens'])
    set_t = set(row['t_tokens'])
    if len(set_q) == 0: return pd.Series([0.0, 0.0])
    
    intersect = len(set_q.intersection(set_t))
    union = len(set_q.union(set_t))
    
    overlap = intersect / len(set_q)
    jaccard = intersect / (union + 1e-9)
    return pd.Series([overlap, jaccard])

df[['overlap_score', 'jaccard_score']] = df.apply(calc_metrics, axis=1)

# 5. HITUNG BM25 (KING OF KEYWORDS)
print("Menghitung BM25 Score...")
corpus_tokens = df['t_tokens'].tolist()
bm25 = BM25Okapi(corpus_tokens)

bm25_scores = []
# Kita pakai enumerate agar indexnya pasti integer urut (0, 1, 2...)
# Ini penting agar cocok dengan urutan corpus_tokens
for idx, row in tqdm(df.iterrows(), total=len(df)):
    # PERBAIKAN DISINI: Hapus kurung siku di sekitar row['q_tokens']
    score = bm25.get_batch_scores(row['q_tokens'], [idx])[0]
    bm25_scores.append(score)

df['bm25_score'] = bm25_scores

# 6. SIMPAN
# Drop kolom temporary
df_final = df.drop(columns=['q_tokens', 't_tokens'])
OUTPUT_FINAL = 'dataset_xgboost_FULL_FEATURES.csv'
df_final.to_csv(OUTPUT_FINAL, index=False)

print(f"\nDATASET FINAL SIAP: {OUTPUT_FINAL}")
print(df_final.head(3))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


✅ Stopwords Indonesia loaded.


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


📂 Memproses 170372 baris data...
🧹 Tokenizing & Cleaning (Removing Stopwords)...
🧮 Menghitung Overlap, Jaccard, & Length Ratio...
👑 Menghitung BM25 Score...


  0%|          | 0/170372 [00:00<?, ?it/s]


✅ DATASET FINAL SIAP: dataset_xgboost_FULL_FEATURES.csv
                                               query  \
0  Mengapa orang kafir mendapat azab yang keras d...   
1  Mengapa semua tumbuhan dan pepohonan tunduk ke...   
2  Dalil mengenai tantangan kaum musyrik kepada R...   

                                                text  label  sbert_sim  len_q  \
0  Kekalahan mereka di dunia bukan akhir segalany...      0   0.482377      5   
1  dan tetumbuhan tak berbatang dan pepohonan ber...      1   0.746225      4   
2  Seandainya orang kafir itu mengetahui dengan p...      0   0.445828      8   

   len_t  len_ratio  overlap_score  jaccard_score  bm25_score  
0     17   0.294118           0.40       0.111111    5.370979  
1      6   0.666667           0.50       0.285714   14.956163  
2     16   0.500000           0.25       0.100000    8.676357  
